INSPECTING STREET NETWORK

In [22]:
import geopandas as gpd

# 1. IDs to exclude (unchanged)
EXCLUDE_IDS = {
    58251, 58267, 27, 53086, 56194, 58345, 10148, 53324, 9712,
    62, 70, 58260, 12, 52736, 29566, 28997, 58347, 51655,
    58339, 58341, 58237, 58238, 22084, 41173
}

# 2. File paths (UPDATED)
top150_path  = r"C:\Users\Agustin\Documents\2025summer\trees\top1000\results\top150_high_utci_segments_1000treesGLOBAL.gpkg"
network_path = r"C:\Users\Agustin\Documents\2025summer\trees\top1000\results\street_network1000trees_with_counts_and_length.gpkg"

# 3. Load
top150  = gpd.read_file(top150_path)
network = gpd.read_file(network_path)[['seg_id', 'length_m', 'count']]

# 4. Exclude unwanted segments from both sets
top150  = top150[~top150['seg_id'].isin(EXCLUDE_IDS)]
network = network[~network['seg_id'].isin(EXCLUDE_IDS)]

# 5. Rename the network 'count' so there’s no ambiguity
network = network.rename(columns={'count': 'trip_count'})

# 6. Merge onto the Top-150 (now without the excluded IDs)
top150_full = top150.merge(network, on='seg_id', how='left')

# 7. Verify columns
print("Columns after merge:", list(top150_full.columns))

# 8. Compute basic metrics
total_length_top150_km  = top150_full['length_m'].sum() / 1000
total_length_network_km = network['length_m'].sum() / 1000
pct_length              = (total_length_top150_km / total_length_network_km) * 100

total_trips_top150      = top150_full['trip_count'].sum()
total_trips_network     = network['trip_count'].sum()
pct_trips               = (total_trips_top150 / total_trips_network) * 100

# ─── NEW: compute heat-exposed km (trip_count * length_m) ───────────────
heatex_km_top150 = (top150_full['trip_count'] * top150_full['length_m']).sum() / 1000
heatex_km_net    = (network      ['trip_count'] * network      ['length_m']).sum() / 1000
pct_heatex_km    = (heatex_km_top150 / heatex_km_net) * 100
# ─────────────────────────────────────────────────────────────────────────

# 9. Print results
print(f"Top-150 segments total length:    {total_length_top150_km:.2f} km")
print(f"Entire network total length:      {total_length_network_km:.2f} km")
print(f"% of network length:              {pct_length:.2f}%\n")

print(f"Top-150 segments trip count:       {total_trips_top150}")
print(f"Entire network trip count:        {total_trips_network}")
print(f"% of all counts:                   {pct_trips:.2f}%\n")

print(f"Top-150 heat-exposed km:           {heatex_km_top150:,.1f} km")
print(f"Entire network heat-exposed km:    {heatex_km_net:,.1f} km")
print(f"% of heat-exposed km by Top-150:   {pct_heatex_km:.2f}%")


Columns after merge: ['seg_id', 'utci', 'count', 'geometry', 'length_m', 'trip_count']
Top-150 segments total length:    29.15 km
Entire network total length:      10089.78 km
% of network length:              0.29%

Top-150 segments trip count:       6502335
Entire network trip count:        39292183
% of all counts:                   16.55%

Top-150 heat-exposed km:           1,480,927.3 km
Entire network heat-exposed km:    6,749,548.6 km
% of heat-exposed km by Top-150:   21.94%


COMPARING RESULTS

IDEA 1

In [27]:
import pandas as pd

data = {
    "Scenario": [
        "Base",
        "3°C (150)",  "5°C (150)",  "10°C (150)",
        "3°C (400)",  "5°C (400)",  "10°C (400)",
        "3°C (1000)", "5°C (1000)", "10°C (1000)",
        "1000 trees",  # NEW
    ],
    "Full-Day ≥32°C (%)": [
        57.7, 57.0, 56.0, 52.3, 56.8, 55.2, 50.3, 55.4, 52.2, 43.7,
        54.7,  # NEW
    ],
    "Full-Day ≥38°C (%)": [
        24.4, 20.3, 18.5, 16.2, 19.2, 16.7, 13.8, 15.6, 12.0, 8.9,
        9.9,  # NEW
    ],
    "Core-Day ≥32°C (%)": [
        77.1, 76.2, 74.9, 69.9, 75.9, 73.8, 67.2, 74.0, 69.8, 58.4,
        73.1,  # NEW
    ],
    "Core-Day ≥38°C (%)": [
        32.6, 27.2, 24.7, 21.6, 25.7, 22.3, 18.5, 20.8, 16.1, 11.9,
        13.2,  # NEW
    ],
}

df = pd.DataFrame(data).set_index("Scenario")

# Optional: pretty print with 1 decimal place
with pd.option_context("display.float_format", "{:.1f}".format):
    print(df.to_string())


             Full-Day ≥32°C (%)  Full-Day ≥38°C (%)  Core-Day ≥32°C (%)  Core-Day ≥38°C (%)
Scenario                                                                                   
Base                       57.7                24.4                77.1                32.6
3°C (150)                  57.0                20.3                76.2                27.2
5°C (150)                  56.0                18.5                74.9                24.7
10°C (150)                 52.3                16.2                69.9                21.6
3°C (400)                  56.8                19.2                75.9                25.7
5°C (400)                  55.2                16.7                73.8                22.3
10°C (400)                 50.3                13.8                67.2                18.5
3°C (1000)                 55.4                15.6                74.0                20.8
5°C (1000)                 52.2                12.0                69.8         

IDEA 2

In [31]:
import pandas as pd

data = {
    "Metric": [
        "Median (°C)",
        "90th percentile (°C)",
        "95th percentile (°C)",
    ],
    "Base":          [33.8, 39.1, 39.6],
    "3°C (150)":     [33.4, 38.9, 39.4],
    "5°C (150)":     [33.2, 38.8, 39.4],
    "10°C (150)":    [32.5, 38.7, 39.3],
    "3°C (400)":     [33.3, 38.8, 39.3],
    "5°C (400)":     [33.0, 38.6, 39.3],
    "10°C (400)":    [32.0, 38.5, 39.1],
    "3°C (1000)":    [33.0, 38.5, 39.1],
    "5°C (1000)":    [32.4, 38.2, 38.9],
    "10°C (1000)":   [30.9, 37.8, 38.7],
    "1000 trees":    [32.9, 38.0, 38.8],  # NEW
}

# Build DataFrame and set Metric as index
df = pd.DataFrame(data).set_index("Metric")

# Transpose so scenarios become rows
df_t = df.T
df_t.index.name = "Scenario"

print(df_t.to_string())


Metric       Median (°C)  90th percentile (°C)  95th percentile (°C)
Scenario                                                            
Base                33.8                  39.1                  39.6
3°C (150)           33.4                  38.9                  39.4
5°C (150)           33.2                  38.8                  39.4
10°C (150)          32.5                  38.7                  39.3
3°C (400)           33.3                  38.8                  39.3
5°C (400)           33.0                  38.6                  39.3
10°C (400)          32.0                  38.5                  39.1
3°C (1000)          33.0                  38.5                  39.1
5°C (1000)          32.4                  38.2                  38.9
10°C (1000)         30.9                  37.8                  38.7
1000 trees          32.9                  38.0                  38.8


IDEA 3

In [39]:
import pandas as pd

data = {
    "Metric": [
        "Top 1% exposure (%)",
        "Top 5% exposure (%)",
        "Top 10% exposure (%)",
    ],
    "Base":        [34.0, 57.0, 70.8],
    "3°C (150)":   [31.4, 54.2, 68.8],
    "5°C (150)":   [27.5, 51.0, 66.5],
    "10°C (150)":  [19.1, 44.9, 62.0],
    "3°C (400)":   [31.7, 54.0, 68.2],
    "5°C (400)":   [27.7, 50.7, 65.5],
    "10°C (400)":  [19.2, 44.6, 61.0],
    "3°C (1000)":  [31.6, 53.4, 67.3],
    "5°C (1000)":  [27.3, 49.4, 63.9],
    "10°C (1000)": [17.6, 42.2, 58.2],
    "1000 trees":  [30.1, 53.1, 67.1],  # NEW
}

df = pd.DataFrame(data).set_index("Metric")
print(df.to_string())


                      Base  3°C (150)  5°C (150)  10°C (150)  3°C (400)  5°C (400)  10°C (400)  3°C (1000)  5°C (1000)  10°C (1000)  1000 trees
Metric                                                                                                                                         
Top 1% exposure (%)   34.0       31.4       27.5        19.1       31.7       27.7        19.2        31.6        27.3         17.6        30.1
Top 5% exposure (%)   57.0       54.2       51.0        44.9       54.0       50.7        44.6        53.4        49.4         42.2        53.1
Top 10% exposure (%)  70.8       68.8       66.5        62.0       68.2       65.5        61.0        67.3        63.9         58.2        67.1


IDEA 4

In [40]:
import pandas as pd

data = {
    "Metric": [
        "Overall average (°C)",
        "Daytime average (°C)",
    ],
    "Base":            [32.3, 35.1],
    "3°C (150)":       [31.9, 34.7],
    "5°C (150)":       [31.6, 34.4],
    "10°C (150)":      [30.8, 33.6],
    "3°C (400)":       [31.8, 34.6],
    "5°C (400)":       [31.4, 34.2],
    "10°C (400)":      [30.5, 33.2],
    "3°C (1000)":      [31.5, 34.2],
    "5°C (1000)":      [30.9, 33.7],
    "10°C (1000)":     [29.5, 32.2],
    "1000 trees":      [31.3, 33.7],  # NEW
}

# Build DataFrame and set Metric as index
df = pd.DataFrame(data).set_index("Metric")

# Transpose so scenarios become rows
df_t = df.T
df_t.index.name = "Scenario"

print(df_t.to_string())


Metric       Overall average (°C)  Daytime average (°C)
Scenario                                               
Base                         32.3                  35.1
3°C (150)                    31.9                  34.7
5°C (150)                    31.6                  34.4
10°C (150)                   30.8                  33.6
3°C (400)                    31.8                  34.6
5°C (400)                    31.4                  34.2
10°C (400)                   30.5                  33.2
3°C (1000)                   31.5                  34.2
5°C (1000)                   30.9                  33.7
10°C (1000)                  29.5                  32.2
1000 trees                   31.3                  33.7


In [17]:
import pandas as pd
from pathlib import Path

# 1) Define your scenario directories and labels, now including the Top-1000 runs
scenarios = {
    "Base": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\utci_results"),
    },
    # NEW: 1000 trees (Top-1000 segments)
    "1000 trees": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\trees\top1000\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\trees\top1000\utci_results"),
    },
    "3 °C (150)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\3degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\3degrees\utci_results"),
    },
    "5 °C (150)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\5degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\5degrees\utci_results"),
    },
    "10 °C (150)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\10degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\10degrees\utci_results"),
    },
    "3 °C (400)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top400\3degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top400\3degrees\utci_results"),
    },
    "5 °C (400)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top400\5degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top400\5degrees\utci_results"),
    },
    "10 °C (400)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top400\10degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top400\10degrees\utci_results"),
    },
    # ==== Top-1000 runs ====
    "3 °C (1000)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\utci_results"),
    },
    "5 °C (1000)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\5degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\5degrees\utci_results"),
    },
    "10 °C (1000)": {
        "ana": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\10degrees\analysis"),
        "utci": Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\10degrees\utci_results"),
    },
}

# 2) Master list of hours from Base
base_hourly = pd.read_csv(scenarios["Base"]["ana"] / "hourly_weighted_avg_utci.csv")
if "hour_int" not in base_hourly.columns:
    base_hourly["hour_int"] = base_hourly["hour"].str[:2].astype(int)
master = (
    base_hourly[["hour_int"]]
    .drop_duplicates()
    .sort_values("hour_int")
    .reset_index(drop=True)
)

# 3) Compute km_ridden for Base
kms = []
for hh in master["hour_int"]:
    fp = scenarios["Base"]["utci"] / f"mean_utci_{hh:02d}00.csv"
    if fp.exists():
        total_m = pd.read_csv(fp)["trip_len_m"].sum()
        kms.append(int(round(total_m / 1000)))
    else:
        kms.append(0)
master["km_ridden"] = kms

# 4) Merge in each scenario’s weighted UTCI (only Base keeps the “ UTCI” suffix)
for label, paths in scenarios.items():
    df = pd.read_csv(paths["ana"] / "hourly_weighted_avg_utci.csv")
    if "hour_int" not in df.columns:
        df["hour_int"] = df["hour"].str[:2].astype(int)
    new_col = f"{label} UTCI" if label == "Base" else label
    master = master.merge(
        df[["hour_int", "weighted_utci"]].rename(columns={"weighted_utci": new_col}),
        on="hour_int",
        how="left"
    )

# 5) Formatting
master = master.rename(columns={"hour_int": "Hour"})
master["Hour"] = master["Hour"].apply(lambda h: f"{h:02d}")
master["km_ridden"] = master["km_ridden"].map("{:,}".format)

# Round every column except Hour and km_ridden to 2 decimals
for col in master.columns:
    if col not in ["Hour", "km_ridden"]:
        master[col] = master[col].astype(float).map("{:.2f}".format)

# 6) Select & order columns (dict preserves insertion order)
cols = ["Hour", "km_ridden", "Base UTCI"] + [label for label in scenarios if label != "Base"]
master = master[cols]

# 7) Print
print(master.to_string(index=False))


Hour km_ridden Base UTCI 1000 trees 3 °C (150) 5 °C (150) 10 °C (150) 3 °C (400) 5 °C (400) 10 °C (400) 3 °C (1000) 5 °C (1000) 10 °C (1000)
  00   229,517     23.07      23.44      22.70      22.46       21.84      22.60      22.29       21.50       22.33       21.84        20.61
  01   137,180     22.81      23.18      22.44      22.20       21.58      22.34      22.03       21.24       22.07       21.58        20.34
  02    90,774     22.00      22.36      21.63      21.39       20.79      21.53      21.22       20.45       21.27       20.79        19.58
  03    56,928     21.10      21.44      20.75      20.52       19.95      20.66      20.36       19.63       20.41       19.95        18.79
  04    48,890     20.67      20.99      20.33      20.11       19.56      20.24      19.96       19.26       20.00       19.56        18.46
  05    90,408     20.39      20.73      20.02      19.78       19.17      19.92      19.62       18.84       19.65       19.15        17.92
  06   246,35

In [45]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# —————————————————————————————————————————————————————————————————————
# IDs to drop entirely (consistent across all scenarios)
EXCLUDE_IDS = {
    58251, 58267, 27, 53086, 56194, 58345, 10148, 53324, 9712,
    62, 70, 58260, 12, 52736, 29566, 28997, 58347, 51655,
    58339, 58341, 58237, 58238, 22084, 41173
}

# Reference Top-1000 list (keep as your current file)
TOP1000_GPKG = Path(
    r"C:\Users\Agustin\Documents\2025summer\sanity_check\results\temp_segments\top1000_high_utci_segments_neat_GLOBAL.gpkg"
)

# Scenario street-network files (Base, Cooling, and Tree-planting)
scenario_paths_1000 = {
    "Base": Path(
        r"C:\Users\Agustin\Documents\2025summer\sanity_check\results\temp_segments\street_network_with_counts_and_length.gpkg"
    ),
    "3 °C (1000)": Path(
        r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\3degrees\results\3degrees_top1000_street_network_with_counts_and_length.gpkg"
    ),
    "5 °C (1000)": Path(
        r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\5degrees\results\5degrees_top1000_street_network_with_counts_and_length.gpkg"
    ),
    "10 °C (1000)": Path(
        r"C:\Users\Agustin\Documents\2025summer\sanity_check\cooling_scenario\top1000\10degrees\results\10degrees_top1000_street_network_with_counts_and_length.gpkg"
    ),
    "1000 trees": Path(
        r"C:\Users\Agustin\Documents\2025summer\trees\top1000\results\street_network1000trees_with_counts_and_length.gpkg"
    ),
}
# —————————————————————————————————————————————————————————————————————

# 1) Load the reference Top-1000 seg_id list
if not TOP1000_GPKG.exists():
    raise FileNotFoundError(f"Top-1000 GPKG not found: {TOP1000_GPKG}")

top1000_ids = set(gpd.read_file(TOP1000_GPKG)["seg_id"].astype(int).tolist())
top1000_ids -= EXCLUDE_IDS

# 2) Summarize function
def summarize(path, seg_ids, exclude_ids):
    if not path.exists():
        raise FileNotFoundError(f"Scenario file not found: {path}")
    
    gdf = gpd.read_file(path).set_index("seg_id")
    gdf = gdf.loc[~gdf.index.isin(exclude_ids)]

    # Ensure dtypes
    gdf["count"]    = gdf["count"].astype(int)
    gdf["length_m"] = gdf["length_m"].astype(float)

    # Totals for the whole network
    total_counts = int(gdf["count"].sum())
    total_km     = (gdf["count"] * gdf["length_m"]).sum() / 1000.0

    # Subset to reference Top-1000
    top = gdf.loc[gdf.index.intersection(seg_ids)]
    top_counts = int(top["count"].sum())
    top_km     = (top["count"] * top["length_m"]).sum() / 1000.0

    return total_counts, top_counts, total_km, top_km

# 3) Run through all scenarios
results_1000 = []
for label, gpkg_path in scenario_paths_1000.items():
    tot_c, top_c, tot_km, top_km = summarize(gpkg_path, top1000_ids, EXCLUDE_IDS)
    pct_c  = (top_c / tot_c) if tot_c else float("nan")
    pct_km = (top_km / tot_km) if tot_km else float("nan")
    results_1000.append({
        "Scenario":       label,
        "Total counts":   tot_c,
        "Top1000 counts": top_c,
        "% counts":       f"{pct_c:.2%}" if pd.notna(pct_c) else "—",
        "Total km":       tot_km,
        "Top1000 km":     top_km,
        "% km":           f"{pct_km:.2%}" if pd.notna(pct_km) else "—",
    })

# 4) Build DataFrame and order rows
order = ["Base", "3 °C (1000)", "5 °C (1000)", "10 °C (1000)", "1000 trees"]
df1000 = pd.DataFrame(results_1000).set_index("Scenario").loc[order].reset_index()

# Format for readability
df1000["Total counts"]   = df1000["Total counts"].map("{:,}".format)
df1000["Top1000 counts"] = df1000["Top1000 counts"].map("{:,}".format)
df1000["Total km"]       = df1000["Total km"].map("{:,.1f}".format)
df1000["Top1000 km"]     = df1000["Top1000 km"].map("{:,.1f}".format)

# 5) Print or export summary
print(df1000.to_string(index=False))
# df1000.to_csv("top1000_consistent_summary.csv", index=False)


    Scenario Total counts Top1000 counts % counts    Total km  Top1000 km   % km
        Base   48,233,408     23,583,637   48.89% 8,329,736.2 4,278,187.7 51.36%
 3 °C (1000)   40,868,080     16,218,309   39.68% 7,153,834.9 3,102,286.5 43.37%
 5 °C (1000)   36,652,736     12,002,965   32.75% 6,252,854.4 2,201,305.9 35.20%
10 °C (1000)   24,935,909        286,138    1.15% 4,079,953.6    28,405.1  0.70%
  1000 trees   39,292,183     12,427,728   31.63% 6,749,548.6 2,409,325.4 35.70%


HEAT-EXPOSED KM RIDDEN SUMMARY

In [46]:
import pandas as pd

# 1) Enter your “Total km” results from the screenshots, now including Top-1000 runs + 1000 trees
data = {
    "Scenario": [
        "Base",
        "3 °C (150)", "5 °C (150)", "10 °C (150)",
        "3 °C (400)", "5 °C (400)", "10 °C (400)",
        "3 °C (1000)", "5 °C (1000)", "10 °C (1000)",
        "1000 trees",  # NEW
    ],
    "Total_km": [
        8_329_736.2,  # Base
        7_780_722.9,  # 3°C (150)
        7_230_708.2,  # 5°C (150)
        6_058_876.2,  # 10°C (150)
        7_600_801.5,  # 3°C (400)
        6_919_266.9,  # 5°C (400)
        5_400_327.9,  # 10°C (400)
        7_153_834.9,  # 3°C (1000)
        6_252_854.4,  # 5°C (1000)
        4_079_953.6,  # 10°C (1000)
        6_749_548.6,  # 1000 trees  ← NEW
    ],
}

df = pd.DataFrame(data).set_index("Scenario")

# 2) Compute reductions relative to Base
base = df.at["Base", "Total_km"]
df["Δ km"] = base - df["Total_km"]
df["Δ %"]  = (df["Δ km"] / base * 100).round(1)

# 3) Format for readability
df = df.assign(
    **{
        "Total km":     df["Total_km"].map("{:,.1f}".format),
        "Reduction km": df["Δ km"].map("{:,.1f}".format),
        "Reduction %":  df["Δ %"].map("{:.1f}%".format),
    }
)[["Total km", "Reduction km", "Reduction %"]]

# 4) Print the summary
print(df.to_string())


                 Total km Reduction km Reduction %
Scenario                                          
Base          8,329,736.2          0.0        0.0%
3 °C (150)    7,780,722.9    549,013.3        6.6%
5 °C (150)    7,230,708.2  1,099,028.0       13.2%
10 °C (150)   6,058,876.2  2,270,860.0       27.3%
3 °C (400)    7,600,801.5    728,934.7        8.8%
5 °C (400)    6,919,266.9  1,410,469.3       16.9%
10 °C (400)   5,400,327.9  2,929,408.3       35.2%
3 °C (1000)   7,153,834.9  1,175,901.3       14.1%
5 °C (1000)   6,252,854.4  2,076,881.8       24.9%
10 °C (1000)  4,079,953.6  4,249,782.6       51.0%
1000 trees    6,749,548.6  1,580,187.6       19.0%
